In [3]:
import pandas as pd
import json

# Load the Excel file
file_path = "mess_menu.xlsx"  # Update with your actual file path
df = pd.read_excel(file_path, dtype=str)  # Read all data as strings

# Remove rows that contain ONLY asterisks or have `*` anywhere in any column
df = df[~df.apply(lambda row: row.astype(str).str.contains(r"^\**$", regex=True, na=False) |  # Entire cell is only "*"
                             row.astype(str).str.contains(r"\*", regex=True, na=False), axis=1)]  # Any "*" in the cell
df.head()


# Create a dictionary to store the structured menu
menu_dict = {}

# Loop through each day's column (excluding the first column which contains meal types)
for column in df.columns[1:]:  
    day_menu = {}
    
    # Identify meal type rows
    for meal_type in ["BREAKFAST", "LUNCH", "DINNER"]:
        meal_rows = df[df.iloc[:, 0].str.contains(meal_type, na=False, case=False)].index

        # If meal_rows is not empty, extract the meal items
        if not meal_rows.empty:
            start_index = meal_rows[0] + 1  # Start from the row after "BREAKFAST", "LUNCH", or "DINNER"
            meal_items = []
            
            # Collect all food items until the next meal type appears
            for i in range(start_index, len(df)):
                if df.iloc[i, 0] in ["BREAKFAST", "LUNCH", "DINNER"]:  
                    break  # Stop if the next meal type appears
                
                food_item = df.iloc[i][column]
                if pd.notna(food_item):  # Ignore NaN values
                    meal_items.append(food_item.strip())

            # Store meal items if they exist
            if meal_items:
                day_menu[meal_type] = meal_items

    # Store the day's menu if it contains any meals
    if day_menu:
        menu_dict[column] = day_menu  

# Convert to JSON
json_output = json.dumps(menu_dict, indent=4)

# Save to a file
with open("mess_menu_final_2.json", "w") as json_file:
    json_file.write(json_output)

print("JSON file created successfully")

JSON file created successfully
